# 🏋️‍♂️ Human Activity Recognition (HAR) - R3D-18 Kaggle GPU Trainer
This notebook trains a **3D ResNet-18 (R3D-18)** Spatiotemporal CNN on your custom action video dataset using **Kaggle's Free NVIDIA T4 (16GB VRAM) GPU**.

### Instructions:
1. In Notebook Settings (right sidebar) -> **Accelerator -> GPU T4 x2 or GPU P100**
2. Add your uploaded dataset via **+ Add Data** (or upload your `custom_dataset.zip`)
3. Set `DATASET_PATH` in Cell 2 below to point to your unzipped dataset folder.
4. Click **Run All**.
5. When finished, download `har_r3d18_custom_model.pth` and import it into your HAR Platform!

In [ ]:
!pip install --upgrade pip
!pip install opencv-python-headless torchvision torchaudio

In [ ]:
import os
import re
import time
import json
import random
import zipfile
from pathlib import Path
from typing import Optional, Tuple, List

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models.video as video_models

# ---------------- CONFIGURATION ----------------
# Update this if your dataset is zipped or under /kaggle/input/
DATASET_INPUT_DIR = "/kaggle/input"
WORK_DIR = Path("/kaggle/working")
EXTRACT_DIR = WORK_DIR / "dataset_extracted"

NUM_FRAMES = 16
RESOLUTION = 224        # High quality 224x224 (Made possible by Kaggle 16GB GPU!)
BATCH_SIZE = 8          # T4 16GB handles batch_size=8 at 224x224 easily
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using compute device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Find dataset files (zip or directory)
found_zips = list(Path(DATASET_INPUT_DIR).rglob("*.zip"))
if found_zips:
    target_zip = found_zips[0]
    print(f"Found dataset zip: {target_zip}. Extracting...")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    DATASET_ROOT = EXTRACT_DIR
else:
    # Check if folders already extracted in input
    DATASET_ROOT = Path(DATASET_INPUT_DIR)

# Find folders containing class videos
classes = []
for p in DATASET_ROOT.rglob("*"):
    if p.is_dir() and not p.name.startswith(".") and p.name != "dataset_extracted":
        vids = list(p.glob("*.mp4")) + list(p.glob("*.avi")) + list(p.glob("*.mov"))
        if len(vids) >= 5:
            if p.name not in classes:
                classes.append(p.name)

classes = sorted(classes)
print(f"Detected {len(classes)} action classes:", classes)

In [ ]:
# Preprocessing: Aspect-ratio preserving center crop to target resolution
def resize_short_edge_and_center_crop(frame: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    h, w = frame.shape[:2]
    if h == target_h and w == target_w:
        return frame
    if h <= w:
        scale = target_h / h
        new_h = target_h
        new_w = max(int(round(w * scale)), target_w)
    else:
        scale = target_w / w
        new_w = target_w
        new_h = max(int(round(h * scale)), target_h)

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    y_off = (new_h - target_h) // 2
    x_off = (new_w - target_w) // 2
    return resized[y_off: y_off + target_h, x_off: x_off + target_w]

def sample_video_frames(video_path: Path, num_frames=16, resolution=(224, 224)) -> Optional[torch.Tensor]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    try:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            return None
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
        sampled = []
        curr = 0
        for target_idx in indices:
            if target_idx > curr + 2:
                cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
                curr = target_idx
            while curr < target_idx:
                cap.grab()
                curr += 1
            ret, frame = cap.read()
            curr += 1
            if not ret or frame is None:
                if sampled:
                    sampled.append(sampled[-1].copy())
                else:
                    sampled.append(np.zeros((resolution[0], resolution[1], 3), dtype=np.uint8))
            else:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                sampled.append(resize_short_edge_and_center_crop(rgb, resolution[0], resolution[1]))

        while len(sampled) < num_frames:
            sampled.append(sampled[-1].copy())

        arr = np.stack(sampled, axis=0).astype(np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(3, 0, 1, 2) # [C, T, H, W]
        # Normalize Kinetics
        mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
        return (tensor - mean) / std
    finally:
        cap.release()

In [ ]:
class VideoHARClassificationDataset(Dataset):
    def __init__(self, samples: List[Tuple[Path, int]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = sample_video_frames(path, num_frames=NUM_FRAMES, resolution=(RESOLUTION, RESOLUTION))
        if tensor is None:
            tensor = torch.zeros((3, NUM_FRAMES, RESOLUTION, RESOLUTION), dtype=torch.float32)
        return tensor, label

# Collect samples
all_samples = []
class_to_idx = {c: i for i, c in enumerate(classes)}
for c in classes:
    for folder in DATASET_ROOT.rglob(c):
        if folder.is_dir():
            vids = list(folder.glob("*.mp4")) + list(folder.glob("*.avi")) + list(folder.glob("*.mov"))
            for v in vids:
                all_samples.append((v, class_to_idx[c]))

print(f"Total video samples found: {len(all_samples)}")
random.seed(42)
random.shuffle(all_samples)

train_len = int(0.8 * len(all_samples))
val_len = len(all_samples) - train_len
train_samples = all_samples[:train_len]
val_samples = all_samples[train_len:]

train_ds = VideoHARClassificationDataset(train_samples)
val_ds = VideoHARClassificationDataset(val_samples)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Model Definition: Pretrained R3D-18
from torchvision.models.video import r3d_18, R3D_18_Weights

print("Loading pretrained R3D-18 weights (Kinetics-400)...")
model = r3d_18(weights=R3D_18_Weights.DEFAULT)

# Freeze early layers (partial fine-tuning)
for name, param in model.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

# Replace classification head
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(classes))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("Model ready for training!")

In [ ]:
# Training Loop
best_val_acc = 0.0
save_model_path = WORK_DIR / "har_r3d18_custom_model.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    start_t = time.time()
    for videos, labels in train_loader:
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(videos)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * videos.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    epoch_train_loss = train_loss / total if total > 0 else 0
    epoch_train_acc = (correct / total * 100) if total > 0 else 0

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = model(videos)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * videos.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total if val_total > 0 else 0
    epoch_val_acc = (val_correct / val_total * 100) if val_total > 0 else 0
    epoch_time = time.time() - start_t

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.1f}s) | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.1f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.1f}%")

    # Save best model checkpoint in format compatible with our local HAR platform
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        checkpoint = {
            "architecture": "r3d_18",
            "classes": classes,
            "model_state_dict": model.state_dict(),
            "accuracy": float(epoch_val_acc / 100.0),
            "config": {
                "resolution": RESOLUTION,
                "num_frames": NUM_FRAMES,
                "batch_size": BATCH_SIZE,
            },
            "timestamp": time.time()
        }
        torch.save(checkpoint, save_model_path)
        print(f"  ⭐ Best model updated (Val Acc: {best_val_acc:.2f}%) -> {save_model_path.name}")

print(f"\nTraining complete! Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Download '{save_model_path.name}' from Kaggle Output to use on your local platform!")